Install Required Packages

In [1]:
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.16
    Uninstalling langchain-protocol-0.0.16:
      Successfully uninstalled langchain-protocol-0.0.16
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source o

In [27]:
from google.colab import files

uploaded = files.upload()

In [27]:
import os

print(os.listdir())

In [27]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Itinerary.pdf")

documents = loader.load()

print("Number of pages:", len(documents))
print(documents[0].page_content[:500])

Splitting the text

In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

In [27]:
for i, chunk in enumerate(chunks):
    print(f"\n----- Chunk {i+1} -----")
    print(chunk.page_content[:200])

In [28]:
!pip install sentence-transformers

In [28]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [28]:
vector = embedding_model.embed_query(
    chunks[0].page_content
)

print(type(vector))
print(len(vector))

creating the vector database

In [28]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

print("Vector database created successfully!")

In [28]:
query = "What is this document about?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print(doc.page_content)
    print("-"*50)

In [28]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

print("FAISS created successfully")

install Gemini SDK

In [18]:
!pip install -q google-generativeai

Configure Gemini

In [29]:
import google.generativeai as genai

genai.configure(
    api_key="YOUR_API_KEY"
)

model = genai.GenerativeModel("gemini-2.5-flash")

In [29]:
response = model.generate_content(
    "What is Artificial Intelligence?"
)

print(response.text)

In [29]:
question = "What is this document about?"

# Retrieve top 3 chunks
docs = vectorstore.similarity_search(
    question,
    k=3
)

context = "\n".join(
    [doc.page_content for doc in docs]
)

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{question}
"""

response = model.generate_content(prompt)

print(response.text)

In [25]:
vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

In [29]:
question = "Summarize this document"

docs = vectorstore.similarity_search(
    question,
    k=3
)

context = "\n".join(
    [doc.page_content for doc in docs]
)

prompt = f"""
Answer only from the provided context.

Context:
{context}

Question:
{question}
"""

response = model.generate_content(prompt)

print(response.text)